# Spark SQL — Motor Cars Dataset (mcar.txt)
### Pure PySpark SQL: Filter | GroupBy | Aggregation | Select

## Step 0 — Download winutils + Spark Setup

In [3]:
import os, sys, ssl, urllib.request

# ── Download correct 64-bit winutils ─────────────────────────────────────────
os.makedirs('hadoop/bin', exist_ok=True)
ctx  = ssl._create_unverified_context()
base = 'https://github.com/cdarlint/winutils/raw/master/hadoop-3.3.6/bin/'

for f in ['winutils.exe', 'hadoop.dll']:
    dest = f'hadoop/bin/{f}'
    urllib.request.urlretrieve(base + f, dest)
    print(f'✅ {f} → {os.path.getsize(dest)} bytes')

# ── Environment variables — MUST be absolute path ────────────────────────────
os.environ['JAVA_HOME']      = r'C:\Users\yrghimire\.jdks\corretto-1.8.0_482'
os.environ['HADOOP_HOME']    = os.path.abspath('hadoop')   # ✅ absolute path
os.environ['PATH']           = os.path.abspath('hadoop/bin') + ';' + os.environ['PATH']
os.environ['PYSPARK_PYTHON'] = sys.executable
os.makedirs(r'C:\tmp\hive', exist_ok=True)                 # ✅ required by Hadoop

print('HADOOP_HOME:', os.environ['HADOOP_HOME'])

✅ winutils.exe → 119296 bytes
✅ hadoop.dll → 78848 bytes
HADOOP_HOME: c:\Users\yrghimire\Chapter\spring_2026\cse817\bigdata_act\chap2\hadoop


In [2]:
from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField,
                                StringType, DoubleType, IntegerType)

try:
    SparkContext.getOrCreate().stop()
except:
    pass

conf = (SparkConf()
        .setAppName('SparkSQL_MCar')
        .setMaster('local[*]')
        .set('spark.driver.host',            'localhost')
        .set('spark.driver.memory',          '4g')
        .set('spark.sql.codegen.wholeStage', 'false'))

sc    = SparkContext(conf=conf)
spark = SparkSession.builder.getOrCreate()
sc.setLogLevel('ERROR')

os.makedirs('output/SparkSQL_MCar', exist_ok=True)
print('✅ Spark ready:', spark.version)

✅ Spark ready: 3.5.3


## Step 1 — Save mcar.txt

## Step 2 — (a) Create and Display Spark DataFrame

In [3]:
schema = StructType([
    StructField('car',  StringType(),  True),
    StructField('mpg',  DoubleType(),  True),
    StructField('cyl',  IntegerType(), True),
    StructField('disp', DoubleType(),  True),
    StructField('hp',   IntegerType(), True),
    StructField('drat', DoubleType(),  True),
    StructField('wt',   DoubleType(),  True),
    StructField('qsec', DoubleType(),  True),
    StructField('vs',   IntegerType(), True),
    StructField('am',   IntegerType(), True),
    StructField('gear', IntegerType(), True),
    StructField('carb', IntegerType(), True)
])

df = (spark.read
      .option('header',    'true')
      .option('delimiter', ',')
      .option('nullValue',  '')
      .schema(schema)
      .csv('Dataset/mcar.txt'))

df.printSchema()
print(f'Rows: {df.count()} | Cols: {len(df.columns)}')
df.show(32, truncate=False)

# Register as SQL TempView
df.createOrReplaceTempView('cars')
print('✅ TempView cars registered')

root
 |-- car: string (nullable = true)
 |-- mpg: double (nullable = true)
 |-- cyl: integer (nullable = true)
 |-- disp: double (nullable = true)
 |-- hp: integer (nullable = true)
 |-- drat: double (nullable = true)
 |-- wt: double (nullable = true)
 |-- qsec: double (nullable = true)
 |-- vs: integer (nullable = true)
 |-- am: integer (nullable = true)
 |-- gear: integer (nullable = true)
 |-- carb: integer (nullable = true)

Rows: 32 | Cols: 12
+-------------------+----+---+-----+---+----+-----+-----+---+---+----+----+
|car                |mpg |cyl|disp |hp |drat|wt   |qsec |vs |am |gear|carb|
+-------------------+----+---+-----+---+----+-----+-----+---+---+----+----+
|Mazda RX4          |21.0|6  |160.0|110|3.9 |2.62 |16.46|0  |1  |4   |4   |
|Mazda RX4 Wag      |21.0|6  |160.0|110|3.9 |2.875|17.02|0  |1  |4   |4   |
|Datsun 710         |22.8|4  |108.0|93 |3.85|2.32 |18.61|1  |1  |4   |1   |
|Hornet 4 Drive     |21.4|6  |258.0|110|3.08|3.215|19.44|1  |0  |3   |1   |
|Hornet Sportab

## Step 3 — (b) Filter: MPG less than 18

In [4]:
# DataFrame API
print('=== DataFrame API: mpg < 18 ===')
df.filter(F.col('mpg') < 18).select('car','mpg','cyl','gear').show(truncate=False)

# Spark SQL
print('=== Spark SQL: mpg < 18 ===')
low_mpg_sql = spark.sql("""
    SELECT car, mpg, cyl, gear
    FROM   cars
    WHERE  mpg < 18
    ORDER  BY mpg ASC
""")
low_mpg_sql.show(truncate=False)
print(f'Cars with mpg < 18: {low_mpg_sql.count()}')

# ✅ Save using Spark write — file:/// absolute path fixes Hadoop on Windows
out_b = 'file:///' + os.path.abspath('output/SparkSQL_MCar/b_mpg_less_than_18').replace('\\','/')
low_mpg_sql.coalesce(1).write.format('csv').mode('overwrite').option('header','true').save(out_b)
print('✅ (b) saved')

=== DataFrame API: mpg < 18 ===
+-------------------+----+---+----+
|car                |mpg |cyl|gear|
+-------------------+----+---+----+
|Duster 360         |14.3|8  |3   |
|Merc 280C          |17.8|6  |4   |
|Merc 450SE         |16.4|8  |3   |
|Merc 450SL         |17.3|8  |3   |
|Merc 450SLC        |15.2|8  |3   |
|Cadillac Fleetwood |10.4|8  |3   |
|Lincoln Continental|10.4|8  |3   |
|Chrysler Imperial  |14.7|8  |3   |
|Dodge Challenger   |15.5|8  |3   |
|AMC Javelin        |15.2|8  |3   |
|Camaro Z28         |13.3|8  |3   |
|Ford Pantera L     |15.8|8  |5   |
|Maserati Bora      |15.0|8  |5   |
+-------------------+----+---+----+

=== Spark SQL: mpg < 18 ===
+-------------------+----+---+----+
|car                |mpg |cyl|gear|
+-------------------+----+---+----+
|Cadillac Fleetwood |10.4|8  |3   |
|Lincoln Continental|10.4|8  |3   |
|Camaro Z28         |13.3|8  |3   |
|Duster 360         |14.3|8  |3   |
|Chrysler Imperial  |14.7|8  |3   |
|Maserati Bora      |15.0|8  |5   |
|Me

## Step 4 — (c) Average Weight by Cylinders

In [5]:
# DataFrame API
print('=== DataFrame API: Avg weight by cylinders ===')
avg_wt_df = (df
    .groupBy('cyl')
    .agg(
        F.round(F.avg('wt'), 3).alias('avg_weight'),
        F.round(F.min('wt'), 3).alias('min_weight'),
        F.round(F.max('wt'), 3).alias('max_weight'),
        F.count('car')         .alias('num_cars')
    )
    .orderBy('cyl')
)
avg_wt_df.show(truncate=False)

# Spark SQL
print('=== Spark SQL: Avg weight by cylinders ===')
avg_wt_sql = spark.sql("""
    SELECT cyl,
           ROUND(AVG(wt), 3) AS avg_weight,
           ROUND(MIN(wt), 3) AS min_weight,
           ROUND(MAX(wt), 3) AS max_weight,
           COUNT(car)        AS num_cars
    FROM   cars
    GROUP  BY cyl
    ORDER  BY cyl
""")
avg_wt_sql.show(truncate=False)

# ✅ Save using Spark write
out_c = 'file:///' + os.path.abspath('output/SparkSQL_MCar/c_avg_weight_by_cyl').replace('\\','/')
avg_wt_sql.coalesce(1).write.format('csv').mode('overwrite').option('header','true').save(out_c)
print('✅ (c) saved')

=== DataFrame API: Avg weight by cylinders ===
+---+----------+----------+----------+--------+
|cyl|avg_weight|min_weight|max_weight|num_cars|
+---+----------+----------+----------+--------+
|4  |2.286     |1.513     |3.19      |11      |
|6  |3.117     |2.62      |3.46      |7       |
|8  |3.999     |3.17      |5.424     |14      |
+---+----------+----------+----------+--------+

=== Spark SQL: Avg weight by cylinders ===
+---+----------+----------+----------+--------+
|cyl|avg_weight|min_weight|max_weight|num_cars|
+---+----------+----------+----------+--------+
|4  |2.286     |1.513     |3.19      |11      |
|6  |3.117     |2.62      |3.46      |7       |
|8  |3.999     |3.17      |5.424     |14      |
+---+----------+----------+----------+--------+

✅ (c) saved


## Step 5 — (d) Select Gear where Cylinders >= 4 and <= 6

In [6]:
# DataFrame API
print('=== DataFrame API: gear where 4 <= cyl <= 6 ===')
gear_df = (df
    .filter((F.col('cyl') >= 4) & (F.col('cyl') <= 6))
    .select('car', 'cyl', 'gear')
    .orderBy('cyl', 'gear')
)
gear_df.show(truncate=False)

# Spark SQL
print('=== Spark SQL: gear where 4 <= cyl <= 6 ===')
gear_sql = spark.sql("""
    SELECT car, cyl, gear
    FROM   cars
    WHERE  cyl BETWEEN 4 AND 6
    ORDER  BY cyl, gear
""")
gear_sql.show(truncate=False)
print(f'Total: {gear_sql.count()}')

# Gear distribution
print('Gear distribution within cyl 4-6:')
spark.sql("""
    SELECT cyl, gear, COUNT(*) AS count
    FROM   cars
    WHERE  cyl BETWEEN 4 AND 6
    GROUP  BY cyl, gear
    ORDER  BY cyl, gear
""").show()

# ✅ Save using Spark write
out_d = 'file:///' + os.path.abspath('output/SparkSQL_MCar/d_gear_by_cyl_4_to_6').replace('\\','/')
gear_sql.coalesce(1).write.format('csv').mode('overwrite').option('header','true').save(out_d)
print('✅ (d) saved')

=== DataFrame API: gear where 4 <= cyl <= 6 ===
+--------------+---+----+
|car           |cyl|gear|
+--------------+---+----+
|Toyota Corona |4  |3   |
|Datsun 710    |4  |4   |
|Merc 240D     |4  |4   |
|Merc 230      |4  |4   |
|Fiat 128      |4  |4   |
|Honda Civic   |4  |4   |
|Toyota Corolla|4  |4   |
|Fiat X1-9     |4  |4   |
|Volvo 142E    |4  |4   |
|Porsche 914-2 |4  |5   |
|Lotus Europa  |4  |5   |
|Hornet 4 Drive|6  |3   |
|Valiant       |6  |3   |
|Mazda RX4     |6  |4   |
|Mazda RX4 Wag |6  |4   |
|Merc 280      |6  |4   |
|Merc 280C     |6  |4   |
|Ferrari Dino  |6  |5   |
+--------------+---+----+

=== Spark SQL: gear where 4 <= cyl <= 6 ===
+--------------+---+----+
|car           |cyl|gear|
+--------------+---+----+
|Toyota Corona |4  |3   |
|Datsun 710    |4  |4   |
|Merc 240D     |4  |4   |
|Merc 230      |4  |4   |
|Fiat 128      |4  |4   |
|Honda Civic   |4  |4   |
|Toyota Corolla|4  |4   |
|Fiat X1-9     |4  |4   |
|Volvo 142E    |4  |4   |
|Porsche 914-2 |4  |5  

## Step 6 — Summary Statistics via Spark SQL

In [7]:
print('=== Overall Dataset Summary ===')
spark.sql("""
    SELECT COUNT(*)          AS total_cars,
           ROUND(AVG(mpg),2) AS avg_mpg,
           MIN(mpg)          AS min_mpg,
           MAX(mpg)          AS max_mpg,
           ROUND(AVG(wt), 2) AS avg_weight,
           ROUND(AVG(hp), 2) AS avg_hp
    FROM   cars
""").show(truncate=False)

print('=== Cars per Cylinder ===')
spark.sql("""
    SELECT cyl, COUNT(*) AS total, ROUND(AVG(mpg),2) AS avg_mpg
    FROM   cars
    GROUP  BY cyl
    ORDER  BY cyl
""").show()

=== Overall Dataset Summary ===
+----------+-------+-------+-------+----------+------+
|total_cars|avg_mpg|min_mpg|max_mpg|avg_weight|avg_hp|
+----------+-------+-------+-------+----------+------+
|32        |20.09  |10.4   |33.9   |3.22      |146.69|
+----------+-------+-------+-------+----------+------+

=== Cars per Cylinder ===
+---+-----+-------+
|cyl|total|avg_mpg|
+---+-----+-------+
|  4|   11|  26.66|
|  6|    7|  19.74|
|  8|   14|   15.1|
+---+-----+-------+



## Step 7 — Final Summary & Stop Spark

In [8]:
print('='*60)
print('   SPARK SQL — mcar.txt — TASK SUMMARY')
print('='*60)
print(f'  (a) Rows: {df.count()} | Cols: {len(df.columns)}')
print(f'  (b) Cars with mpg < 18      : {df.filter(F.col("mpg") < 18).count()}')
print(f'  (c) Cylinder groups         : {avg_wt_df.count()}')
print(f'  (d) Cars with cyl 4-6       : {gear_df.count()}')
print()
print('  Output saved to: output/SparkSQL_MCar/')
print('    ├── b_mpg_less_than_18/')
print('    ├── c_avg_weight_by_cyl/')
print('    └── d_gear_by_cyl_4_to_6/')
print('='*60)

sc.stop()
print('\n✅ Spark stopped. All done!')

   SPARK SQL — mcar.txt — TASK SUMMARY
  (a) Rows: 32 | Cols: 12
  (b) Cars with mpg < 18      : 13
  (c) Cylinder groups         : 3
  (d) Cars with cyl 4-6       : 18

  Output saved to: output/SparkSQL_MCar/
    ├── b_mpg_less_than_18/
    ├── c_avg_weight_by_cyl/
    └── d_gear_by_cyl_4_to_6/

✅ Spark stopped. All done!
